# Re-evaluate checkpoints on the corrected val/test split, with efficiency numbers

`src/data.py`'s `FlowerDataModule.setup()` had a bug: `val_set` and `test_set` were both built from
`train_subset`, so every metric logged under the `flower-classification-v2` experiment (including the
two models pushed to the HF Hub, both logged at ~1.0) was measured on training data. The split logic
is now fixed, and `corrected-split-retrain` runs are trustworthy.

This notebook re-scores every candidate checkpoint on the real, disjoint val and test sets and
measures inference cost on `cuda:0` (RTX 5070), so the leaked-era and corrected-era models can be
compared on the same footing before deciding what to publish.

In [ ]:
import sqlite3
import sys
import time
from pathlib import Path

import lightning as L
import pandas as pd
import torch

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))

from train.config import PRETRAINED_MODEL_REGISTRY  # noqa: E402
from src.classifier import FlowerClassifier  # noqa: E402
from src.data import FlowerDataModule, FlowerDataset  # noqa: E402

CKPT_DIR = BASE_DIR / "models_checkpoints"
DATA_ROOT = BASE_DIR / "data"
DB_PATH = BASE_DIR / "mlflow.db"
DEVICE = torch.device("cuda:0")  # RTX 5070

# Short run ids; checkpoints are named {model}-epoch={n}-val_acc={acc}-{run_id[:8]}.ckpt,
# and the architecture + experiment come from mlflow.db, so nothing is duplicated here.
RUN_IDS = [
    # flower-classification-v2 (pre-fix, logged metrics leaked)
    "6899e8e2",
    "d6508546",
    "4632c1bb",
    "747435c3",
    # corrected-split-retrain: best run per architecture
    "01d3e87a",
    "620728ca",
    "a6094159",
]

print(torch.cuda.get_device_name(DEVICE))

In [ ]:
def resolve(run_id: str) -> dict:
    """Short run id -> architecture, experiment name, checkpoint path."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        row = conn.execute(
            "SELECT r.run_uuid, e.name AS experiment FROM runs r "
            "JOIN experiments e ON e.experiment_id = r.experiment_id "
            "WHERE r.run_uuid LIKE ?",
            (f"{run_id}%",),
        ).fetchone()
        if row is None:
            raise ValueError(f"no run {run_id!r} in {DB_PATH}")
        arch = conn.execute(
            "SELECT value FROM params WHERE run_uuid = ? AND key = 'pretrained_model'",
            (row["run_uuid"],),
        ).fetchone()[0]
    finally:
        conn.close()

    ckpts = list(CKPT_DIR.glob(f"*-{run_id}.ckpt"))
    if len(ckpts) != 1:
        raise FileNotFoundError(
            f"expected exactly 1 checkpoint for {run_id}, found {ckpts}"
        )

    return {
        "run_id": run_id,
        "architecture": arch,
        "experiment": row["experiment"],
        "ckpt": ckpts[0],
    }


specs = [resolve(run_id) for run_id in RUN_IDS]
for s in specs:
    print(f"{s['run_id']}  {s['architecture']:<18} {s['experiment']}")

In [7]:
dataset = FlowerDataset(DATA_ROOT)
num_classes = len(dataset.classes)

# same seed/split ratios FlowerDataModule used during training (train/run_training.py
# only overrides data_root and batch_size), so this reproduces the intended val/test sets
dm = FlowerDataModule(DATA_ROOT, batch_size=32)
dm.setup("fit")
dm.setup("test")

print(f"train={len(dm.train_set)} val={len(dm.val_set)} test={len(dm.test_set)}")

train=5733 val=1228 test=1228


In [8]:
def load_model(spec: dict) -> FlowerClassifier:
    factory, head_name, _ = PRETRAINED_MODEL_REGISTRY[spec["architecture"]]
    model = FlowerClassifier.load_from_checkpoint(
        spec["ckpt"],
        pretrained_model=factory(),
        num_classes=num_classes,
        class_weights=None,
        head_name=head_name,
        class_names=dataset.classes,
        map_location="cpu",
        strict=False,  # checkpoint has a criterion.weight buffer we don't restore for inference
    )
    model.eval()
    return model

## Size and latency on `cuda:0`

Batch size 1, 224x224 — single-image latency, which is what the API serves. `torch.cuda.synchronize`
after every forward pass is what makes these numbers real: CUDA kernel launches are async, so timing
without it measures the launch, not the inference.

In [ ]:
@torch.no_grad()
def efficiency(model: torch.nn.Module, ckpt_path: Path) -> dict:
    # ponytail: batch-1, single process, models timed back-to-back, so thermal/clock drift
    # shows up as ~1ms of run-to-run noise. Use nsight or an isolated per-model run if you
    # need to resolve differences smaller than that.
    model = model.to(DEVICE).eval()
    x = torch.randn(1, 3, 224, 224, device=DEVICE)

    for _ in range(10):  # warmup
        model(x)
    torch.cuda.synchronize(DEVICE)

    latencies = []
    for _ in range(50):
        start = time.perf_counter()
        model(x)
        torch.cuda.synchronize(
            DEVICE
        )  # CUDA is async: without this we time the launch, not the run
        latencies.append((time.perf_counter() - start) * 1000)
    latencies.sort()

    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

    return {
        "num_parameters": sum(p.numel() for p in model.parameters()),
        "model_size_mb": (param_bytes + buffer_bytes) / 1e6,  # deployable state_dict
        "checkpoint_size_mb": ckpt_path.stat().st_size
        / 1e6,  # .ckpt also carries optimizer state
        "latency_ms_mean": sum(latencies) / len(latencies),
        "latency_ms_p95": latencies[int(len(latencies) * 0.95)],
    }

## Score every checkpoint

In [ ]:
trainer = L.Trainer(
    accelerator="gpu", devices=[0], logger=False, enable_checkpointing=False
)

results = []
for spec in specs:
    model = load_model(spec)
    val_metrics = trainer.validate(model, datamodule=dm, verbose=False)[0]
    test_metrics = trainer.test(model, datamodule=dm, verbose=False)[0]
    results.append(
        {
            "run_id": spec["run_id"],
            "architecture": spec["architecture"],
            "experiment": spec["experiment"],
            **val_metrics,
            **test_metrics,
            **efficiency(model, spec["ckpt"]),
        }
    )
    del model
    torch.cuda.empty_cache()  # each backbone is up to ~350 MB of weights

results

In [ ]:
df = pd.DataFrame(results)
df.to_csv("published_models_corrected_eval.csv", index=False)
df.sort_values("test_f1", ascending=False)